In [28]:
import pandas as pd
import time
import numpy as np

In [30]:
import pandas as pd
import time

# ==================== 1. 读取数据 ====================
file_path = r'index_optimize.csv'
df = pd.read_csv(file_path, encoding='utf-8', parse_dates=['下单时间'])
df = pd.concat([df] * 10000, ignore_index=True)
print(f"测试数据量：{len(df)} 行\n")

# ==============================================
# 测试 1：查询 用户ID = U001 （公平对比）
# ==============================================
# 无索引
start = time.time()
_ = df[df['用户ID'] == 'U001']
t1_raw = time.time() - start

# 有索引
df_user = df.set_index('用户ID')
start = time.time()
_ = df_user.loc['U001']
t1_idx = time.time() - start

# ==============================================
# 测试 2：查询 订单ID = ORD005 （公平对比）
# ==============================================
# 无索引
start = time.time()
_ = df[df['订单ID'] == 'ORD005']
t2_raw = time.time() - start

# 有索引
df_order = df.set_index('订单ID')
start = time.time()
_ = df_order.loc['ORD005']
t2_idx = time.time() - start

# ==============================================
# 测试 3：查询 2024-05-02 订单（公平对比）
# ==============================================
# 无索引
start = time.time()
_ = df[df['下单时间'].dt.date == pd.to_datetime('2024-05-02').date()]
t3_raw = time.time() - start

# 有索引（已修复，不报错）
df_time = df.set_index('下单时间').sort_index()
start = time.time()
_ = df_time.loc['2024-05-02':'2024-05-02']
t3_idx = time.time() - start

# ==================== 最终输出（绝对公平） ====================
print("="*60)
print("                📊 索引优化公平对比结果")
print("="*60)
print(f"【1】用户ID查询 U001")
print(f"  无索引：{t1_raw:.4f}s  |  索引：{t1_idx:.4f}s  |  提速 {t1_raw/t1_idx:.4f}倍")
print(f"【2】订单ID查询 ORD005")
print(f"  无索引：{t2_raw:.4f}s  |  索引：{t2_idx:.4f}s  |  提速 {t2_raw/t2_idx:.4f}倍")
print(f"【3】日期查询 2024-05-02")
print(f"  无索引：{t3_raw:.4f}s  |  索引：{t3_idx:.4f}s  |  提速 {t3_raw/t3_idx:.4f}倍")
print("="*60)

测试数据量：100000 行

                📊 索引优化公平对比结果
【1】用户ID查询 U001
  无索引：0.0117s  |  索引：0.0184s  |  提速 0.6361倍
【2】订单ID查询 ORD005
  无索引：0.0102s  |  索引：0.0182s  |  提速 0.5578倍
【3】日期查询 2024-05-02
  无索引：0.0245s  |  索引：0.0012s  |  提速 19.8614倍


In [31]:
print("\n5. 索引优化核心规则：")
# 验证索引唯一性（唯一索引查询最快）
print(f"订单ID是否唯一：{df['订单ID'].is_unique} → 适合做唯一索引")
print(f"用户ID是否唯一：{df['用户ID'].is_unique} → 适合做非唯一索引")
# 索引类型选择建议
index_rules = pd.DataFrame({'字段类型': ['唯一值字段（订单ID/用户ID）', '时间字段（下单时间）', '多条件查询字段', '低基数字段（城市）'],'推荐索引': ['单唯一索引', '时间索引', '复合索引', '不建议做索引'],'提速效果': ['10~100倍', '5~20倍', '5~50倍', '无提升（甚至变慢）']})
print("\n索引选型建议：")
print(index_rules)


5. 索引优化核心规则：
订单ID是否唯一：False → 适合做唯一索引
用户ID是否唯一：False → 适合做非唯一索引

索引选型建议：
               字段类型    推荐索引       提速效果
0  唯一值字段（订单ID/用户ID）   单唯一索引    10~100倍
1        时间字段（下单时间）    时间索引      5~20倍
2           多条件查询字段    复合索引      5~50倍
3         低基数字段（城市）  不建议做索引  无提升（甚至变慢）


In [32]:
print("\n6. 索引避坑点：")
# 反例1：索引列做运算 → 索引失效
df_index1['订单ID_new'] = df_index1.index.str.replace('ORD', '')
start_time = time.time()
res_bad1 = df_index1[df_index1['订单ID_new'] == '005']  # 索引列运算后查询，失效
bad1_time = time.time() - start_time
print(f"❌ 索引列运算后查询耗时：{bad1_time:.4f} 秒（索引失效，速度回退）")


6. 索引避坑点：
❌ 索引列运算后查询耗时：0.0206 秒（索引失效，速度回退）
